# Beach quality

The most recent E. coli readings at Toronto's 10 beaches, turned into a GeoJSON file that Leaflet can map.

```
data/torontowater.csv  ->  this notebook  ->  data/beaches.geojson  ->  site/  ->  a map in your browser
```

## Part 1. Load

Each row of the CSV is one water sample: a beach, a sampling site at that beach, a date, an E. coli count and the site's location.

The empty `eColi` cells are already read as `NaN` (pandas's missing value), so there is no `"blank"` to replace as there was in Lab 2.

In [1]:
import pandas as pd
import json
import numpy as np

df = pd.read_csv("../data/Toronto EColi Data/torontowater.csv")

print("rows read:", len(df))
print("missing eColi readings:", df["eColi"].isna().sum())
df.head()

rows read: 102534
missing eColi readings: 3543


,_id,beachId,beachName,siteName,collectionDate,eColi,geometry
0,1,1,Marie Curtis Park East Beach,29W,2026-09-16,NaN,"{""coordinates"": [[-79.539696, 43.58592]], ""typ..."
1,2,1,Marie Curtis Park East Beach,33W,2026-09-16,NaN,"{""coordinates"": [[-79.540886, 43.58486]], ""typ..."
2,3,1,Marie Curtis Park East Beach,32W,2026-09-16,NaN,"{""coordinates"": [[-79.54056, 43.58511]], ""type..."
3,4,1,Marie Curtis Park East Beach,31W,2026-09-16,NaN,"{""coordinates"": [[-79.54039, 43.585304]], ""typ..."
4,5,1,Marie Curtis Park East Beach,30W,2026-09-16,NaN,"{""coordinates"": [[-79.540054, 43.58561]], ""typ..."


## Part 2. Clean

1. `pd.to_datetime` turns the date text into real dates, so "most recent" compares them as dates rather than as text.
2. `dropna(subset=["eColi"])` drops every row with no E. coli reading. **This matters:** every sample from the newest date in the file, 2026-09-16, is empty. Without this step every beach's latest average would be `NaN`.
3. `.copy()` makes `readings` a separate table, so adding columns later doesn't trigger a pandas warning.

In [2]:
df["collectionDate"] = pd.to_datetime(df["collectionDate"])

readings = df.dropna(subset=["eColi"]).copy()

# Get lon and lat for every reading
coords = readings["geometry"].apply(lambda g: json.loads(g)["coordinates"][0])
readings["lon"] = coords.str[0]
readings["lat"] = coords.str[1]

# One row per sampling site, and each beach's centre point
sites = readings.drop_duplicates(["beachId", "siteName"])[["beachId", "beachName", "siteName", "lon", "lat"]]
centres = sites.groupby("beachId")[["lon", "lat"]].median()

def distance_km(lon1, lat1, lon2, lat2):
    """Distance in km between two points on the Earth (the haversine formula)."""
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    a = np.sin((lat2 - lat1) / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2)**2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

# Go through every site and flag the ones far from their beach's centre
MAX_KM = 1
outliers = []
for _, site in sites.iterrows():
    centre = centres.loc[site["beachId"]]
    km = distance_km(site["lon"], site["lat"], centre["lon"], centre["lat"])
    if km > MAX_KM:
        outliers.append(site["siteName"])
        print(site["beachName"], site["siteName"], round(km, 1), "km from the beach centre")

before = len(readings)
readings = readings[~readings["siteName"].isin(outliers)]
print("dropped", before - len(readings), "readings from", len(outliers), "outlier sites")


print("rows dropped:", len(df) - len(readings), "(no eColi reading)")
print("rows kept:   ", len(readings))

Sunnyside Beach 60W 15.6 km from the beach centre
Gibraltar Point Beach GP6 13.8 km from the beach centre
dropped 220 readings from 2 outlier sites
rows dropped: 3763 (no eColi reading)
rows kept:    98771


## Part 3. Keep only each beach's most recent day

- `groupby("beachId")` sorts the rows into one pile per beach.
- `["collectionDate"].transform("max")` finds the newest date in each pile and writes it back onto every row of that pile. The result is a column the same length as `readings`.
- `readings[readings["collectionDate"] == latest]` keeps only the rows whose date equals their beach's newest date.

`groupby` does the looping over the beaches for us, so there is no `for` loop here.

In [3]:
latest = readings.groupby("beachId")["collectionDate"].transform("max")
recent = readings[readings["collectionDate"] == latest].copy()

print("readings on each beach's most recent day:", len(recent))
recent.head()

readings on each beach's most recent day: 50


,_id,beachId,beachName,siteName,collectionDate,eColi,geometry,lon,lat
104,105,1,Marie Curtis Park East Beach,29W,2026-09-07,140.0,"{""coordinates"": [[-79.539696, 43.58592]], ""typ...",-79.539696,43.585920
105,106,1,Marie Curtis Park East Beach,33W,2026-09-07,340.0,"{""coordinates"": [[-79.540886, 43.58486]], ""typ...",-79.540886,43.584860
106,107,1,Marie Curtis Park East Beach,32W,2026-09-07,200.0,"{""coordinates"": [[-79.54056, 43.58511]], ""type...",-79.540560,43.585110
107,108,1,Marie Curtis Park East Beach,31W,2026-09-07,260.0,"{""coordinates"": [[-79.54039, 43.585304]], ""typ...",-79.540390,43.585304
108,109,1,Marie Curtis Park East Beach,30W,2026-09-07,270.0,"{""coordinates"": [[-79.540054, 43.58561]], ""typ...",-79.540054,43.585610


## Part 4. Get longitude and latitude out of `geometry`

Each `geometry` cell is JSON text like `{"coordinates": [[-79.54, 43.58]], "type": "MultiPoint"}`.

- `.apply(lambda g: ...)` runs a small function on every cell; `g` is one cell's text.
- `json.loads(g)` turns that text into a Python dictionary.
- `["coordinates"][0]` gets the first (and only) point, `[lon, lat]`.
- `.str[0]` and `.str[1]` take longitude and latitude out of each pair. GeoJSON puts longitude first.

In [4]:
recent[["beachName", "siteName", "collectionDate", "eColi", "lon", "lat"]].head()

,beachName,siteName,collectionDate,eColi,lon,lat
104,Marie Curtis Park East Beach,29W,2026-09-07,140.0,-79.539696,43.585920
105,Marie Curtis Park East Beach,33W,2026-09-07,340.0,-79.540886,43.584860
106,Marie Curtis Park East Beach,32W,2026-09-07,200.0,-79.540560,43.585110
107,Marie Curtis Park East Beach,31W,2026-09-07,260.0,-79.540390,43.585304
108,Marie Curtis Park East Beach,30W,2026-09-07,270.0,-79.540054,43.585610


## Part 5. One row per beach

`groupby(["beachId", "beachName"])` keeps the name in the result (each ID has exactly one name). `.agg(...)` makes one column from each `new_name=("column", "function")` pair:

- `avg_eColi`: the mean of that day's readings at the beach
- `n_readings`: how many readings went into the average
- `lon` / `lat`: each beach has 5–6 sampling sites, so their mean puts one point in the middle of the beach
- `date`: the day the average comes from

`as_index=False` keeps `beachId` and `beachName` as ordinary columns.

In [5]:
beaches = recent.groupby(["beachId", "beachName"], as_index=False).agg(
    date=("collectionDate", "max"),
    avg_eColi=("eColi", "mean"),
    n_readings=("eColi", "size"),
    lon=("lon", "mean"),
    lat=("lat", "mean"),
)
beaches

,beachId,beachName,date,avg_eColi,n_readings,lon,lat
0,1,Marie Curtis Park East Beach,2026-09-07,242.000000,5,-79.540317,43.585361
1,2,Sunnyside Beach,2026-09-07,65.000000,4,-79.456833,43.637027
2,3,Hanlan's Point Beach,2026-09-07,12.000000,5,-79.395464,43.618579
3,4,Gibraltar Point Beach,2026-09-07,42.000000,5,-79.382079,43.611777
4,5,Centre Island Beach,2026-09-07,34.000000,5,-79.372902,43.616130
5,6,Ward's Island Beach,2026-09-07,18.000000,5,-79.352658,43.629474
6,7,Cherry Beach,2026-09-07,58.000000,5,-79.343059,43.636266
7,8,Woodbine Beaches,2026-09-07,78.000000,5,-79.303899,43.663177
8,9,Kew Balmy Beach,2026-09-07,23.333333,6,-79.289819,43.668145
9,10,Bluffer's Beach Park,2026-09-07,34.000000,5,-79.225762,43.713583


## Part 6. One row to one feature

The same function as Lab 3, Part 3, with the beach columns in `properties`.

- `float()` and `int()` turn pandas numbers into plain Python numbers, which JSON can store.
- `.strftime("%Y-%m-%d")` turns the date back into text, because JSON has no date type.
- `round(..., 1)` makes averages like 46.666… show as 46.7 in a popup.

In [6]:
def row_to_feature(row):
    """Take one row of the beaches table and return one GeoJSON feature."""
    feature = {
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [float(row["lon"]), float(row["lat"])],   # longitude first, then latitude
        },
        "properties": {
            "beach_id": int(row["beachId"]),
            "beach_name": row["beachName"],
            "date": row["date"].strftime("%Y-%m-%d"),
            "avg_ecoli": round(float(row["avg_eColi"]), 1),
            "n_readings": int(row["n_readings"]),
        },
    }
    return feature

print(json.dumps(row_to_feature(beaches.iloc[0]), indent=2))

{
  "type": "Feature",
  "geometry": {
    "type": "Point",
    "coordinates": [
      -79.5403172,
      43.585360800000004
    ]
  },
  "properties": {
    "beach_id": 1,
    "beach_name": "Marie Curtis Park East Beach",
    "date": "2026-09-07",
    "avg_ecoli": 242.0,
    "n_readings": 5
  }
}


## Part 7. Every row, then write the file

`allow_nan=False` stops with an error if any value is `NaN`, because browsers can't read `NaN` in JSON.

In [7]:
features = []
for _, row in beaches.iterrows():
    features.append(row_to_feature(row))

collection = {"type": "FeatureCollection", "features": features}

out_path = "../data/beaches.geojson"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(collection, f, separators=(",", ":"), allow_nan=False)

print(len(features), "features written to", out_path)

10 features written to ../data/beaches.geojson


## Part 8. Check the file

Reads the file back from disk, the way the browser will, and checks it against the table and against Toronto.

In [8]:
with open("../data/beaches.geojson", encoding="utf-8") as f:
    back = json.load(f)

problems = []
feats = back.get("features", [])
if back.get("type") != "FeatureCollection":
    problems.append("The file is not a FeatureCollection.")
if len(feats) != len(beaches):
    problems.append(f"The file has {len(feats)} features but the beaches table has {len(beaches)} rows.")
lons = [ft["geometry"]["coordinates"][0] for ft in feats]
lats = [ft["geometry"]["coordinates"][1] for ft in feats]
if feats and not all(-79.7 < x < -79.1 for x in lons):
    problems.append("Some longitudes are outside Toronto. Are longitude and latitude swapped?")
if feats and not all(43.5 < y < 43.9 for y in lats):
    problems.append("Some latitudes are outside Toronto. Are longitude and latitude swapped?")
if feats and not all({"beach_name", "avg_ecoli", "date"} <= set(ft["properties"]) for ft in feats):
    problems.append("Some features are missing beach_name, avg_ecoli or date in their properties.")

print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

All checks passed


In [9]:
beaches


,beachId,beachName,date,avg_eColi,n_readings,lon,lat
0,1,Marie Curtis Park East Beach,2026-09-07,242.000000,5,-79.540317,43.585361
1,2,Sunnyside Beach,2026-09-07,65.000000,4,-79.456833,43.637027
2,3,Hanlan's Point Beach,2026-09-07,12.000000,5,-79.395464,43.618579
3,4,Gibraltar Point Beach,2026-09-07,42.000000,5,-79.382079,43.611777
4,5,Centre Island Beach,2026-09-07,34.000000,5,-79.372902,43.616130
5,6,Ward's Island Beach,2026-09-07,18.000000,5,-79.352658,43.629474
6,7,Cherry Beach,2026-09-07,58.000000,5,-79.343059,43.636266
7,8,Woodbine Beaches,2026-09-07,78.000000,5,-79.303899,43.663177
8,9,Kew Balmy Beach,2026-09-07,23.333333,6,-79.289819,43.668145
9,10,Bluffer's Beach Park,2026-09-07,34.000000,5,-79.225762,43.713583


## Part 9. A second layer: each beach's average over its last few seasons

Unlike Part 3 (which keeps only the single most recent day), this builds a second map layer: one number per beach, averaged across its last `RECENT_YEARS` seasons -- a toggleable "recent years" view alongside the "latest reading" view.

Each year is averaged on its own first, *then* those yearly averages are averaged together. That matters: a season with more samples shouldn't quietly outweigh a season with fewer. Averaging every individual reading together in one step would do exactly that.

In [11]:
RECENT_YEARS = 3  # how many of the most recent seasons to average together

readings["year"] = readings["collectionDate"].dt.year

# One row per beach per year: that year's own seasonal average E. coli.
yearly = readings.groupby(["beachId", "beachName", "year"], as_index=False).agg(
    season_avg_eColi=("eColi", "mean"),
    n_readings=("eColi", "size"),
)

# Keep each beach's RECENT_YEARS most recent seasons only
yearly["year_rank"] = yearly.groupby("beachId")["year"].rank(method="first", ascending=False)
recent_seasons = yearly[yearly["year_rank"] <= RECENT_YEARS]

print(f"using up to {RECENT_YEARS} most recent seasons per beach:")
print(recent_seasons.groupby("beachName")["year"].apply(list))

using up to 3 most recent seasons per beach:
beachName
Bluffer's Beach Park            [2024, 2025, 2026]
Centre Island Beach             [2024, 2025, 2026]
Cherry Beach                    [2024, 2025, 2026]
Gibraltar Point Beach           [2024, 2025, 2026]
Hanlan's Point Beach            [2024, 2025, 2026]
Kew Balmy Beach                 [2024, 2025, 2026]
Marie Curtis Park East Beach    [2024, 2025, 2026]
Sunnyside Beach                 [2024, 2025, 2026]
Ward's Island Beach             [2024, 2025, 2026]
Woodbine Beaches                [2024, 2025, 2026]
Name: year, dtype: object


In [12]:
recent_avg = recent_seasons.groupby(["beachId", "beachName"], as_index=False).agg(
    avg_eColi=("season_avg_eColi", "mean"),
    seasons_used=("year", "size"),
)

# Reuse each beach's centre point from Part 2 -- it doesn't move year to year
recent_avg = recent_avg.merge(centres.reset_index(), on="beachId")

recent_avg

,beachId,beachName,avg_eColi,seasons_used,lon,lat
0,1,Marie Curtis Park East Beach,151.714036,3,-79.540390,43.585304
1,2,Sunnyside Beach,154.405253,3,-79.455640,43.637110
2,3,Hanlan's Point Beach,43.724442,3,-79.395250,43.618565
3,4,Gibraltar Point Beach,36.570027,3,-79.381563,43.611878
4,5,Centre Island Beach,125.247432,3,-79.373620,43.615900
5,6,Ward's Island Beach,64.042835,3,-79.352684,43.629517
6,7,Cherry Beach,52.673151,3,-79.343250,43.636200
7,8,Woodbine Beaches,68.869697,3,-79.304340,43.663250
8,9,Kew Balmy Beach,102.409091,3,-79.289507,43.668276
9,10,Bluffer's Beach Park,72.447860,3,-79.225750,43.713610


In [13]:
def recent_row_to_feature(row):
    """Take one row of the recent_avg table and return one GeoJSON feature."""
    return {
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [float(row["lon"]), float(row["lat"])],
        },
        "properties": {
            "beach_id": int(row["beachId"]),
            "beach_name": row["beachName"],
            "avg_ecoli": round(float(row["avg_eColi"]), 1),
            "seasons_used": int(row["seasons_used"]),
        },
    }

recent_features = [recent_row_to_feature(row) for _, row in recent_avg.iterrows()]
recent_collection = {"type": "FeatureCollection", "features": recent_features}

recent_out_path = "../data/beaches_recent_avg.geojson"
with open(recent_out_path, "w", encoding="utf-8") as f:
    json.dump(recent_collection, f, separators=(",", ":"), allow_nan=False)

print(len(recent_features), "features written to", recent_out_path)

10 features written to ../data/beaches_recent_avg.geojson


In [14]:
with open("../data/beaches_recent_avg.geojson", encoding="utf-8") as f:
    back = json.load(f)

problems = []
feats = back.get("features", [])
if back.get("type") != "FeatureCollection":
    problems.append("The file is not a FeatureCollection.")
if len(feats) != len(recent_avg):
    problems.append(f"The file has {len(feats)} features but the table has {len(recent_avg)} rows.")
if feats and not all({"beach_name", "avg_ecoli", "seasons_used"} <= set(ft["properties"]) for ft in feats):
    problems.append("Some features are missing expected properties.")

print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

All checks passed
